In [ ]:
# Reset to inline backend in case interactive cell (cell 4) was run first
try:
    _ip = get_ipython()  # noqa: F821
    if _ip is not None:
        _ip.run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# ══════════════════════════════════════════════════════════════════════════════
# STANDALONE BITCOIN MANUAL-BUBBLE MODEL
# Self-contained: loads data, fits power-law support, then fits bubble_shape()
# to each bubble year the user specifies in BUBBLE_YEARS.  No rolling-slope
# phase classification is performed — you choose the peak years; the model
# determines the rise, plateau, and decay parameters automatically.
#
# Workflow:
#   1. Fit power-law support line to historical data.
#   2. For each year in BUBBLE_YEARS, locate the local log-excess peak inside a
#      ±BUBBLE_YEAR_WINDOW search window and fit bubble_shape() sequentially
#      to the residual log-excess (largest peak first).
#   3. Classify fitted bubbles as MAJOR (top N_MAJOR by peak K) / MINOR.
#   4. Compute composite model R².
#   5. Extrapolate or average future bubble parameters and plot projections.
#
# No other cells need to have run first.
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from scipy.stats import linregress
from matplotlib.ticker import FixedLocator, StrMethodFormatter, NullFormatter
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ── Data & input ──────────────────────────────────────────────────────────────
csv_path       = './BitcoinPricesDaily.csv'
FIT_MIN_DATE   = '2010-07-17'   # exclude early near-zero trades from the fit
FIT_MAX_DATE   = None           # None = use all available data
MIN_DATA_YEARS = 1.0
# Bitcoin had no real market price in its first year (optimal time origin 2009-07-25).
# Excluding it prevents the power-law support fit from being skewed by near-zero trades.

# ── Support line ──────────────────────────────────────────────────────────────
SUPPORT_PERCENTILE = 20   # bottom X% of OLS residuals → support candidates
SUPPORT_QUANTILE   = 0.50

# ── Bubble years ──────────────────────────────────────────────────────────────
BUBBLE_YEARS = [2011, 2013, 2017, 2021, 2025]
# List the calendar years in which you believe a Bitcoin bubble peaked.
# For each year, the model searches for the highest log-excess (price/support)
# within ±BUBBLE_YEAR_WINDOW of Jan 1 of that year.
#
# Tips:
#   • Include separate entries for close double-peaks (e.g. 2013 and 2014 for
#     the April-2013 and December-2013 peaks).
#   • Years with no historical data in their window are automatically skipped
#     (no error).  Predicted bubbles are added via STEP 6 extrapolation.
#   • Order does not matter — the model sorts by peak magnitude before fitting.

BUBBLE_YEAR_WINDOW = 0.75
# Half-width of the search window, in years.
# A value of 1.0 means the model searches within ±1 year of Jan 1 of each
# bubble year, i.e. a 2-year window centred on Jan 1 of that year.
#
# Larger values help when the actual peak falls far from Jan 1 (e.g. a
# December peak in a "year" bubble).  Smaller values prevent adjacent
# bubble years from sharing data points.  Windows may overlap — the
# sequential residual fitting handles this correctly.

# ── Fitting ───────────────────────────────────────────────────────────────────
#FIT_CONTEXT_YR      = 1.0 #default=1.0
FIT_CONTEXT_YR      = 1.0
# Extra data (in years) included beyond the ±BUBBLE_YEAR_WINDOW on each side
# when optimising a bubble's parameters.  This context shows the optimiser
# where the residual returns to zero, anchoring the rise and decay tails.
# Increase if bubble fits look truncated; decrease if adjacent bubbles bleed in.

FIT_RISE_LOOKBACK_YR = 0.75
# The bubble's rise can start this many years before the left edge of the
# search window.  Needed because the actual price rise begins before the
# peak window — the bubble was already building up before the window opens.

PLATEAU_PARALLEL_SUPPORT = True
# True  → plat_pow is fixed at 0 (plateau price grows at exactly the support
#          power-law rate; log-excess is flat during the plateau).  Fewer
#          parameters, more stable fits, consistent with the phase model.
# False → plat_pow is a free parameter (6-D optimisation).  Allows the
#          plateau to tilt: < 0 = rounded top, > 0 = blow-off acceleration.

CAP_COMPOSITE_OVERLAP = True
# When True, the composite (historical + predicted bubbles) is built with
# sequential subtraction: bubbles are processed chronologically and each
# bubble's contribution at any time t is capped to the remaining budget,
# where the budget starts at the historical maximum log-excess.
# This prevents overlapping bubble tails / rises from stacking and driving
# the model price far above the data.  Set to False to restore the original
# uncapped additive sum.

PLAT_POW_RANGE = 8.0
# |plat_pow| upper bound during optimisation and prediction clipping.
# ±slope_sup ≈ ±5.8 spans "price constant" to "double support slope"; 8.0
# gives modest headroom beyond the physically meaningful range.

DE_MAXITER = 2000
# Maximum iterations for the Differential Evolution optimiser per bubble.
# With 5–6 free parameters, 2000 iterations is a good balance of speed and
# solution quality.  Increase to 4000+ if fit costs look suspiciously high.

DE_POPSIZE = 18
# Population multiplier for DE.  Population size = DE_POPSIZE × n_params.
# With 5–6 params, this gives 90–108 candidates per generation.
# DE recommends at least 10–15×; 18 is a safe conservative choice.

# ── Classification ────────────────────────────────────────────────────────────
N_MAJOR = 5
# Top N_MAJOR fitted bubbles by peak K are labelled "major"; the rest "minor".
# Bitcoin's canonical halving-driven cycles: 2011, 2013, 2017, 2021.
# Set to 5 to also include 2025 as a major bubble.

MAX_MAJOR_BUBBLES = None
# After classification, keep only the N highest-K major bubbles in the model.
# None = keep all N_MAJOR.  Example: set to 3 to drop the weakest major.

MAX_MINOR_BUBBLES = 8
# Keep only the N highest-K minor bubbles in the model.
# None = keep all.  Reduces clutter in plots and speeds up prediction.

# ── Prediction ────────────────────────────────────────────────────────────────
N_PREDICT_MAJOR      = 3     # number of future major bubbles to project
N_PREDICT_MINOR      = 1     # number of future minor bubbles to project

PREDICT_MODE         = 'extrap'
# 'avg'    — each predicted bubble gets the weighted-average parameters of the
#            last PREDICT_LAST_N_MAJOR historical bubbles.  All predictions are
#            identical in shape; only the start time shifts.  Stable / conservative.
# 'extrap' — parameters are linearly extrapolated along their historical trend
#            (r and d log-linearly; K, dur_plateau, plat_pow linearly).

PREDICT_LAST_N_MAJOR = 3
# Only the most recent N major bubbles feed the avg / extrapolation.
# None = use all detected major bubbles.
# Setting this to 3-4 focuses on recent cycles and ignores the very different
# early history (e.g. the 2011 bubble with r≈4).

PREDICT_LAST_N_MINOR = 4   # same idea for minor bubbles

MAJOR_INTERVAL_YR        = 3.8
# Fallback inter-bubble interval (years) used when fewer than 2 historical
# intervals can be measured (i.e. only 1 historical major bubble exists).

MAJOR_INTERVAL_USE_TREND = True
# False → predicted interval = weighted average of historical intervals (stable).
# True  → interval is also linearly extrapolated along its historical trend,
#         so predicted cycles can lengthen or shorten over time.

MAJOR_EXTRAP_WEIGHTS     = None
# Per-bubble, per-parameter importance weights for the avg / extrapolation.
# None = uniform (every bubble and every parameter contributes equally).
#
# If set, must be a flat list of length N_major × 6, where N_major is the
# number of historical major bubbles actually used (after PREDICT_LAST_N_MAJOR
# trimming).  The layout is row-major: one row of 6 weights per bubble,
# ordered chronologically (oldest first):
#
#   [w_r, w_d, w_K, w_interval, w_dur_plateau, w_plat_pow]   ← bubble 1 (oldest)
#   [w_r, w_d, w_K, w_interval, w_dur_plateau, w_plat_pow]   ← bubble 2
#   ...
#
# Parameter roles:
#
#   r  (rise rate, yr⁻¹)
#       Controls how steeply price climbs above support during the rise phase.
#       High r → sharp, spike-like rally.  r has historically declined each
#       cycle (early bubbles were explosive; recent ones are more gradual).
#       Extrapolated log-linearly so it can only approach zero, never go negative.
#
#   d  (decay rate, yr⁻¹)
#       Controls how quickly price falls back toward support after the peak.
#       High d → short, sharp crash.  Like r, d has trended downward over time
#       (bear markets are getting longer).  Also log-linear.
#
#   K  (peak log-excess, log₁₀ units)
#       log₁₀(peak_price / support_price_at_peak).  K=1 means the peak was
#       10× the support line; K=0.5 means ~3×.  K has declined each cycle
#       as Bitcoin matures — predicting its future value is the key uncertainty.
#       Extrapolated log-linearly (geometric mean in avg mode): each cycle's K
#       is a roughly constant fraction of the prior one, so log(K) is linear
#       in cycle index.  This also ensures predicted K is always positive.
#       A higher weight on recent bubbles will anchor K to the current regime.
#
#   interval  (time between successive t_rise values, years)
#       How many years elapse between the start of one bubble and the next.
#       Used to position the predicted bubble in time.  The weight column here
#       applies to the *intervals* array (one shorter than the bubble array),
#       so the last bubble's w_interval weight is effectively unused.
#
#   dur_plateau  (plateau duration, years)
#       How long price stays near the peak before the decay begins.  Short
#       plateaus give sharp V-shaped peaks; longer ones give extended tops.
#       Has been roughly stable across cycles, so averaging is usually fine.
#
#   plat_pow  (plateau power-law tilt, dimensionless)
#       Differential exponent during the plateau relative to the support slope.
#       plat_pow=0 → price grows at exactly the support rate (flat log-excess).
#       plat_pow<0 → price drifts down during the plateau (rounded top).
#       plat_pow>0 → price accelerates beyond support growth (blow-off top).
#       Currently fixed at 0 when PLATEAU_PARALLEL_SUPPORT=True.
#
# Example — downweight the oldest bubble and ignore plat_pow entirely:
#   MAJOR_EXTRAP_WEIGHTS = [0.5, 0.5, 0.5, 0.5, 0.5, 0,   # bubble 1 (old)
#                           1.0, 1.0, 1.0, 1.0, 1.0, 0,   # bubble 2
#                           1.5, 1.5, 1.5, 1.5, 1.5, 0]   # bubble 3 (recent)

MINOR_EXTRAP_WEIGHTS     = None  # same layout as MAJOR_EXTRAP_WEIGHTS
MINOR_INTERVAL_USE_TREND = True

# ── Bubble model colours ──────────────────────────────────────────────────────
DATA_COLOR   = '#1D4ED8'   # historical daily price scatter
MAJOR_COLORS = ['#EF4444', '#F97316', '#EAB308', '#22C55E',
                '#3B82F6', '#8B5CF6', '#EC4899', '#14B8A6']
MINOR_COLORS = ['#FCA5A5', '#FDBA74', '#FDE68A', '#86EFAC',
                '#93C5FD', '#C4B5FD', '#F9A8D4', '#99F6E4']

# ── Shared chart colours ──────────────────────────────────────────────────────
SUPPORT_COLOR        = '#3B82F6'   # support power-law line
COMPOSITE_COLOR      = '#DC2626'   # composite model curve
TODAY_COLOR          = '#888888'   # "today" vertical marker
FIT_MIN_COLOR        = '#10B981'   # FIT_MIN_DATE vertical marker
FIT_MAX_COLOR        = '#EF4444'   # FIT_MAX_DATE vertical marker
SCATTER_COLOR        = '#94A3B8'   # raw data scatter in residual / excess plots
EXCESS_SMOOTH_COLOR  = '#2563EB'   # smoothed excess line in decomposition plot
RESID_SUP_COLOR      = '#3B82F6'   # smoothed support-only residual line
RESID_COMP_COLOR     = '#EF4444'   # smoothed composite residual line
REFERENCE_LINE_COLOR = 'black'     # zero-reference horizontal lines
GRID_MAJOR_COLOR     = '#CCCCCC'   # major grid lines
GRID_MINOR_COLOR     = '#E5E5E5'   # minor grid lines
PLOT_BG_COLOR        = 'white'     # axes background colour
FALLBACK_TEXT_COLOR  = 'gray'      # "no data" placeholder text

# ── Plot sizing ───────────────────────────────────────────────────────────────
FIGSIZE_WIDE      = (15, 9)    # log-log, semi-log, and most chart panels
FIGSIZE_DECOMP    = (15, 10)   # bubble decomposition (2 stacked panels)
FIGSIZE_RESIDUALS = (15, 8)    # residual comparison (2 panels)
FIGSIZE_ENVELOPE  = (13, 7)    # peak-aligned envelope chart

# ── Plot axes ─────────────────────────────────────────────────────────────────
PLOT_YEARS_MIN         = 1.0    # first year on the dense plot grid and full x-axis
PLOT_YEARS_MAX         = 72.0   # last year on the dense plot grid and full x-axis
PLOT_GRID_POINTS       = 3000   # number of points in the dense time-grid
PRICE_YMIN             = 0.01   # lower price y-axis limit (USD)
PRICE_YMAX             = 1e8    # upper price y-axis limit (USD)
ZOOM_XMIN              = 15     # zoomed chart left bound (years since genesis)
ZOOM_XMAX              = 35     # zoomed chart right bound (years since genesis)
ZOOM_YMIN              = 2e4    # zoomed chart lower price limit (USD)
ZOOM_YMAX              = 3e7    # zoomed chart upper price limit (USD)
RESIDUAL_SMOOTH_WINDOW = 90     # rolling-mean window (days) for residual plots


# ══════════════════════════════════════════════════════════════════════════════
# LOAD + PREPARE DATA
# ══════════════════════════════════════════════════════════════════════════════
try:
    df = pd.read_csv(csv_path)
    print(f"CSV loaded.  Shape: {df.shape}")
except FileNotFoundError:
    print(f"File not found: {csv_path}")

df.columns = ['Date', 'Price']
df['Date']  = pd.to_datetime(df['Date'], format='%m/%d/%y', errors='coerce')
df = df.dropna(subset=['Date']).sort_values('Date')

x_dates = df['Date']
y_data  = df['Price'].astype(float)

genesis     = pd.to_datetime('2009-07-25')
years_since = (x_dates - genesis).dt.days / 365.25

valid       = y_data > 0
years_valid = years_since[valid].astype(float)
y_valid     = y_data[valid]

reasonable = years_valid >= MIN_DATA_YEARS
years_all  = years_valid[reasonable].values
y_all      = y_valid[reasonable].values
dates_all  = x_dates[valid][reasonable].values

log_t_all = np.log10(years_all)
log_p_all = np.log10(y_all)

fit_mask = np.ones(len(dates_all), dtype=bool)
if FIT_MIN_DATE:
    fit_mask &= pd.to_datetime(dates_all) >= pd.to_datetime(FIT_MIN_DATE)
if FIT_MAX_DATE:
    fit_mask &= pd.to_datetime(dates_all) <= pd.to_datetime(FIT_MAX_DATE)

log_t     = log_t_all[fit_mask]
log_p     = log_p_all[fit_mask]
years_fit = years_all[fit_mask]
dates_fit = dates_all[fit_mask]

today_years   = (pd.to_datetime('today') - genesis).days / 365.25
fit_min_years = (pd.to_datetime(FIT_MIN_DATE) - genesis).days / 365.25 if FIT_MIN_DATE else None
fit_max_years = (pd.to_datetime(FIT_MAX_DATE) - genesis).days / 365.25 if FIT_MAX_DATE else None

print(f"Date range  : {pd.Timestamp(dates_all[0]).date()} → {pd.Timestamp(dates_all[-1]).date()}")
print(f"Fit window  : {fit_mask.sum()} points")
print(f"Price range : ${y_all.min():,.2f} – ${y_all.max():,.2f}")
print(f"Today       : t = {today_years:.3f} yr")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: FIT SUPPORT LINE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print(f"STEP 1: FITTING SUPPORT LINE (bottom {SUPPORT_PERCENTILE}% OLS residual)")
print("=" * 80)

slope_ols, intercept_ols, _, _, _ = linregress(log_t, log_p)
ols_resid    = log_p - (intercept_ols + slope_ols * log_t)
cutoff       = np.percentile(ols_resid, SUPPORT_PERCENTILE)
support_mask = ols_resid <= cutoff

X_support = sm.add_constant(log_t[support_mask])
res_sup   = sm.QuantReg(log_p[support_mask], X_support).fit(q=SUPPORT_QUANTILE, max_iter=10000)

intercept_sup = res_sup.params[0]
slope_sup     = res_sup.params[1]
A_sup         = 10 ** intercept_sup
B_sup         = slope_sup

print(f"  Support points : {support_mask.sum()} / {len(log_t)}")
print(f"  Power law      : price = {A_sup:.4e} × t^{B_sup:.4f}")

log_support_all = intercept_sup + slope_sup * log_t_all
log_support_fit = intercept_sup + slope_sup * log_t

log_excess_all = log_p_all - log_support_all
log_excess_fit = log_p     - log_support_fit


# ══════════════════════════════════════════════════════════════════════════════
# BUBBLE SHAPE FUNCTION  (used by STEP 3 below)
# ══════════════════════════════════════════════════════════════════════════════
def bubble_shape(t, t_rise, r, t_plateau, t_decay, d, plat_pow=0.0):
    """
    Log₁₀ bubble excess above the power-law support line.

    Parameters
    ----------
    t         : array of time values (years since genesis)
    t_rise    : start of the exponential rise
    r         : rise rate (yr⁻¹); price doubles relative to support every log(2)/r years
    t_plateau : end of rise / start of plateau
    t_decay   : end of plateau / start of decay
    d         : decay rate (yr⁻¹)
    plat_pow  : differential power-law exponent during plateau, relative to support.
                price ∝ t^(slope_sup + plat_pow) during [t_plateau, t_decay].
                  plat_pow = 0          → plateau parallels support (log_excess = K)
                  plat_pow = −slope_sup → price is constant during plateau
                  plat_pow > 0          → blow-off top (price grows faster than support)

    Phases
    ------
    Rise    [t_rise, t_plateau):
        log_excess(t) = r·(t − t_rise) + slope_sup·log₁₀(t_rise / t)
        Equivalently: price grows exponentially from support(t_rise).

    Plateau [t_plateau, t_decay):
        log_excess(t) = K + plat_pow·log₁₀(t / t_plateau)
        where K = log_excess(t_plateau) = peak of rise phase.
        plat_pow = 0 recovers the original "plateau parallels support" behaviour.

    Decay   [t_decay, ∞):
        log_excess(t) = K_end − d·(t − t_decay) + slope_sup·log₁₀(t_decay / t), clipped ≥ 0
        where K_end = K + plat_pow·log₁₀(t_decay / t_plateau) = log_excess at t_decay.
        Price decays exponentially back toward support(t_decay).
    """
    t = np.asarray(t, dtype=float)
    result = np.zeros_like(t)
    if t_plateau <= t_rise:
        return result
    # K: log-excess at the start of the plateau (= peak of rise)
    K = r * (t_plateau - t_rise) + slope_sup * np.log10(
        np.maximum(t_rise / t_plateau, 1e-12))
    if K <= 0:
        return result
    # K_end: log-excess at the end of the plateau (= start of decay)
    K_end = (K + plat_pow * np.log10(np.maximum(t_decay / t_plateau, 1e-12))
             if t_decay > t_plateau else K)

    # Rise
    m = (t >= t_rise) & (t < t_plateau)
    if m.any():
        result[m] = np.maximum(
            r * (t[m] - t_rise) + slope_sup * np.log10(
                np.maximum(t_rise / t[m], 1e-12)), 0.0)
    # Plateau (general power law)
    m = (t >= t_plateau) & (t < t_decay)
    if m.any():
        result[m] = np.maximum(
            K + plat_pow * np.log10(np.maximum(t[m] / t_plateau, 1e-12)), 0.0)
    # Decay
    m = t >= t_decay
    if m.any():
        result[m] = np.maximum(
            K_end - d * (t[m] - t_decay) + slope_sup * np.log10(
                np.maximum(t_decay / t[m], 1e-12)), 0.0)
    return result


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: LOCATE BUBBLE PEAKS FROM BUBBLE_YEARS
# ══════════════════════════════════════════════════════════════════════════════
# For each year in BUBBLE_YEARS, find the highest log-excess (price/support)
# within a ±BUBBLE_YEAR_WINDOW window centred on Jan 1 of that year.
#
# The search window is just for locating the peak — the fitting window in
# STEP 3 is wider (extends by ±FIT_CONTEXT_YR beyond the search window) so
# the optimiser sees the residual approaching zero on both sides.
#
# Years where the search window falls entirely outside the available data are
# silently skipped.  They will not produce a fitted bubble — use the STEP 6
# prediction to project those future peaks.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print(f"STEP 2: LOCATE BUBBLE PEAKS  (BUBBLE_YEAR_WINDOW = ±{BUBBLE_YEAR_WINDOW} yr)")
print("=" * 80)
print(f"\n  Searching for peaks near: {BUBBLE_YEARS}")
print(f"\n  {'Year':>4}  {'t_center':>8}  {'Search window':>22}  {'Peak date':>12}  "
      f"{'t_peak':>7}  {'Raw K':>6}")
print("  " + "-" * 72)

bm_peaks = []   # list of dicts describing each located peak

for yr in BUBBLE_YEARS:
    # Jan 1 of the specified year, converted to years since genesis
    t_center = (pd.to_datetime(f'{yr}-01-01') - genesis).days / 365.25
    t_lo     = t_center - BUBBLE_YEAR_WINDOW
    t_hi     = t_center + BUBBLE_YEAR_WINDOW

    # Only search within available data
    window_mask = (years_fit >= t_lo) & (years_fit <= t_hi)
    if not window_mask.any():
        print(f"  {yr:>4}  {t_center:>8.3f}  [{t_lo:>8.3f}, {t_hi:>8.3f}]  "
              f"{'(no data — skipped)':>42}")
        continue

    local_exc = log_excess_fit[window_mask]
    local_yrs = years_fit[window_mask]
    local_dt  = dates_fit[window_mask]
    peak_i    = int(np.argmax(local_exc))
    peak_t    = local_yrs[peak_i]
    peak_K    = local_exc[peak_i]
    peak_date = pd.Timestamp(local_dt[peak_i]).strftime('%Y-%m-%d')

    bm_peaks.append({
        'bubble_year': yr,
        'peak_t':      peak_t,     # years since genesis at the log-excess peak
        'region_lo':   t_lo,       # left edge of search window
        'region_hi':   t_hi,       # right edge of search window
        'raw_K':       peak_K,
    })
    print(f"  {yr:>4}  {t_center:>8.3f}  [{t_lo:>8.3f}, {t_hi:>8.3f}]  "
          f"{peak_date:>12}  {peak_t:>7.3f}  {peak_K:>6.3f}")

if not bm_peaks:
    print("\n  No peaks found in any window.  Check BUBBLE_YEARS and BUBBLE_YEAR_WINDOW.")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3: FIT BUBBLE_SHAPE() TO EACH LOCATED PEAK
# ══════════════════════════════════════════════════════════════════════════════
# Fitting strategy: sequential residual fitting, largest raw peak first.
#
# Each bubble is fitted to the CURRENT residual (raw log-excess minus the
# shapes already fitted).  After fitting, this bubble's shape is subtracted
# from the residual before the next bubble is fitted.  This prevents
# double-counting: without it, all bubbles independently fit the full signal
# and their sum far exceeds the data.
#
# The optimisation window extends FIT_CONTEXT_YR beyond the search window so
# the optimiser sees the residual returning to zero on both sides.  The rise
# start (t_rise) can begin up to FIT_RISE_LOOKBACK_YR before the left edge
# of the search window, because the actual rally precedes the peak window.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("STEP 3: FITTING BUBBLE SHAPES")
print("=" * 80)
print(f"  plat_pow: {'fixed = 0 (plateau parallels support)' if PLATEAU_PARALLEL_SUPPORT else 'free parameter'}")
print(f"  FIT_CONTEXT_YR = {FIT_CONTEXT_YR}  FIT_RISE_LOOKBACK_YR = {FIT_RISE_LOOKBACK_YR}")
print(f"  DE_MAXITER = {DE_MAXITER}  DE_POPSIZE = {DE_POPSIZE}")


def fit_manual_bubble(pk, idx, residual):
    """
    Fit bubble_shape() to one manually-located peak.

    Parameters
    ----------
    pk       : peak dict from bm_peaks (keys: peak_t, region_lo, region_hi, raw_K)
    idx      : integer index for RNG seeding (ensures reproducibility)
    residual : current residual log-excess (raw log-excess minus previously
               fitted bubble contributions)

    Returns
    -------
    dict of bubble parameters, dates, and diagnostics
    """
    region_lo = pk['region_lo']
    region_hi = pk['region_hi']
    peak_t    = pk['peak_t']

    # ── Fitting data window ──────────────────────────────────────────────────
    # Extend beyond the search window so the optimiser sees the return to zero.
    t_lo = max(years_fit[0],  region_lo - FIT_CONTEXT_YR)
    t_hi = min(years_fit[-1], region_hi + FIT_CONTEXT_YR)
    ctx  = (years_fit >= t_lo) & (years_fit <= t_hi)
    t_ctx = years_fit[ctx]
    exc   = np.maximum(0.0, residual[ctx])   # fit to non-negative residual

    span = (region_hi - region_lo) + 2 * FIT_CONTEXT_YR   # total fitting span

    # ── Optimisation bounds ──────────────────────────────────────────────────
    # t_rise: start from FIT_RISE_LOOKBACK_YR before window, up to the peak
    t_rise_lb = max(years_fit[0], region_lo - FIT_RISE_LOOKBACK_YR)

    bounds_5 = [
        (t_rise_lb, peak_t),    # t_rise  — rise begins before or at the peak
        (0.05, 20.0),            # r       — rise rate (yr⁻¹)
        (0.02, span),            # dur_rise — rise duration
        (0.0,  span),            # dur_plateau — 0 = no plateau (V-shaped peak)
        (0.05, 20.0),            # d       — decay rate (yr⁻¹)
    ]

    if PLATEAU_PARALLEL_SUPPORT:
        # 5-D optimisation: plat_pow is fixed at 0
        def obj5(p5):
            tr, r_, dr, dp, d_ = p5
            pred = bubble_shape(t_ctx, tr, r_, tr + dr, tr + dr + dp, d_, 0.0)
            return float(np.sum((exc - pred) ** 2))

        res    = differential_evolution(obj5, bounds_5, maxiter=DE_MAXITER,
                                        popsize=DE_POPSIZE, seed=42 + idx,
                                        tol=1e-10, polish=True)
        params = list(res.x) + [0.0]   # append plat_pow = 0
        cost   = res.fun

    else:
        # 6-D optimisation: plat_pow is a free parameter
        bounds_6 = bounds_5 + [(-PLAT_POW_RANGE, PLAT_POW_RANGE)]

        def obj6(p6):
            tr, r_, dr, dp, d_, pp = p6
            pred = bubble_shape(t_ctx, tr, r_, tr + dr, tr + dr + dp, d_, pp)
            return float(np.sum((exc - pred) ** 2))

        res    = differential_evolution(obj6, bounds_6, maxiter=DE_MAXITER,
                                        popsize=DE_POPSIZE, seed=42 + idx,
                                        tol=1e-10, polish=True)
        params = list(res.x)
        cost   = res.fun

    # ── Unpack and derive K values ───────────────────────────────────────────
    tr, r_, dr, dp, d_, pp = params
    tplat  = tr + dr
    tdec   = tplat + dp
    # K_peak: log-excess at the end of the rise (= start of plateau)
    K_peak = max(r_ * dr + slope_sup * np.log10(max(tr / tplat, 1e-12)), 0.0)
    # K_end: log-excess at the end of the plateau (= start of decay)
    K_end  = K_peak + pp * np.log10(max(tdec / tplat, 1e-12)) if dp > 0 else K_peak
    # K_bubble: the maximum log-excess (peak of the whole bubble)
    K_bub  = max(K_peak, K_end)
    # t_end: approximate time when the decay returns to support (log_excess → 0)
    t_end  = tdec + (max(K_end, 0.0) / d_ if d_ > 0 else 0.0)

    return {
        't_rise':      tr,      'r':           r_,    'dur_rise':    dr,
        't_plateau':   tplat,   'dur_plateau':  dp,   't_decay':     tdec,
        'd':           d_,      'plat_pow':     pp,
        'K':           K_bub,   'K_peak':       K_peak, 'K_end':     K_end,
        't_end':       t_end,   'cost':         cost,
        't_start':     tr,      # alias used by plot helpers
        'bubble_year': pk['bubble_year'],
        'date_rise':   genesis + pd.Timedelta(days=tr    * 365.25),
        'date_plat':   genesis + pd.Timedelta(days=tplat * 365.25),
        'date_decay':  genesis + pd.Timedelta(days=tdec  * 365.25),
        'date_end':    genesis + pd.Timedelta(days=t_end  * 365.25),
    }


# ── Sequential residual fitting: largest raw peak first ──────────────────────
# Sorting by raw_K descending ensures dominant events are captured cleanly
# before smaller ones.  This mirrors how the phase model fits bubbles.
peaks_by_magnitude = sorted(
    enumerate(bm_peaks),
    key=lambda x: x[1]['raw_K'],
    reverse=True
)

residual_bm  = log_excess_fit.copy()   # mutable residual; starts as raw log-excess
bm_fitted    = []                      # fitted bubble parameter dicts

for rank, (orig_idx, pk) in enumerate(peaks_by_magnitude):
    print(f"  Fitting bubble year {pk['bubble_year']} "
          f"({rank + 1}/{len(bm_peaks)}, raw K={pk['raw_K']:.3f}) ...",
          end=' ', flush=True)
    bp = fit_manual_bubble(pk, orig_idx, residual_bm)
    bm_fitted.append(bp)
    # Subtract this bubble's contribution from the residual
    contrib     = bubble_shape(years_fit, bp['t_rise'], bp['r'],
                               bp['t_plateau'], bp['t_decay'], bp['d'],
                               bp.get('plat_pow', 0.0))
    residual_bm = np.maximum(0.0, residual_bm - contrib)
    pp_str = (f"plat_pow={bp['plat_pow']:+.2f}"
              if not PLATEAU_PARALLEL_SUPPORT else "plat_pow=0")
    print(f"K={bp['K']:.3f} ({10**bp['K']:.1f}×)  {pp_str}  "
          f"cost={bp['cost']:.4f}  resid_max={residual_bm.max():.3f}")

# Sort chronologically by rise time for display and classification
bm_fitted.sort(key=lambda b: b['t_rise'])


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4: CLASSIFY FITTED BUBBLES → MAJOR / MINOR
# ══════════════════════════════════════════════════════════════════════════════
# The top N_MAJOR bubbles by peak K are labelled "major"; the rest are "minor".
# Optional caps (MAX_MAJOR_BUBBLES, MAX_MINOR_BUBBLES) further trim each class
# to the strongest K values so plots stay readable.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("STEP 4: MAJOR / MINOR CLASSIFICATION")
print("=" * 80)

n_det   = len(bm_fitted)
n_maj   = min(N_MAJOR, n_det)
by_K    = sorted(range(n_det), key=lambda i: bm_fitted[i]['K'], reverse=True)
maj_set = set(by_K[:n_maj])

bm_major = sorted([bm_fitted[i] for i in maj_set],
                   key=lambda b: b['t_rise'])
bm_minor = sorted([bm_fitted[i] for i in range(n_det) if i not in maj_set],
                   key=lambda b: b['t_rise'])

# Apply optional caps (keep highest-K within each class, re-sort chronologically)
if MAX_MAJOR_BUBBLES is not None and len(bm_major) > MAX_MAJOR_BUBBLES:
    bm_major = sorted(sorted(bm_major, key=lambda b: -b['K'])[:MAX_MAJOR_BUBBLES],
                      key=lambda b: b['t_rise'])
if MAX_MINOR_BUBBLES is not None and len(bm_minor) > MAX_MINOR_BUBBLES:
    bm_minor = sorted(sorted(bm_minor, key=lambda b: -b['K'])[:MAX_MINOR_BUBBLES],
                      key=lambda b: b['t_rise'])

print(f"  {n_det} total  →  {len(bm_major)} MAJOR + {len(bm_minor)} MINOR")
print(f"  (top {N_MAJOR} by peak K = major"
      + (f"; capped at {MAX_MAJOR_BUBBLES}" if MAX_MAJOR_BUBBLES else "")
      + f";  {len(bm_minor)} minor shown"
      + (f" of {n_det - len(maj_set)}"
         if MAX_MINOR_BUBBLES and n_det - len(maj_set) > len(bm_minor) else "")
      + ")")


def print_bm_table(params, label):
    """
    Print a formatted table of fitted bubble parameters.

    Columns: bubble year, rise date, rise rate r (yr⁻¹), rise duration (yr),
    peak K (log₁₀ multiplier above support), price peak as a multiple of support,
    plateau duration (yr), decay rate d (yr⁻¹), optionally plat_pow, and the
    inter-bubble interval Δt_rise (yr) from the previous bubble.
    """
    if not params:
        print(f"\n  {label}: none"); return
    print(f"\n  {label} BUBBLES:")
    hdr = (f"  {'Yr':>4}  {'date_rise':>12}  {'r':>6}  {'dur_rise':>8}  "
           f"{'K':>6}  {'Peak×':>6}  {'dur_plat':>8}  {'d':>6}")
    if not PLATEAU_PARALLEL_SUPPORT:
        hdr += f"  {'plat_pow':>8}"
    hdr += f"  {'Δt_rise':>8}"
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))
    for i, bp in enumerate(params):
        dt  = f"{bp['t_rise'] - params[i-1]['t_rise']:.2f} yr" if i > 0 else ""
        row = (f"  {bp.get('bubble_year','?'):>4}  "
               f"{bp['date_rise'].strftime('%Y-%m-%d'):>12}  "
               f"{bp['r']:>6.3f}  {bp['dur_rise']:>8.3f}  {bp['K']:>6.3f}  "
               f"{10**bp['K']:>6.1f}×  {bp['dur_plateau']:>8.3f}  {bp['d']:>6.3f}")
        if not PLATEAU_PARALLEL_SUPPORT:
            row += f"  {bp['plat_pow']:>+8.3f}"
        row += f"  {dt:>8}"
        print(row)


print_bm_table(bm_major, "MAJOR")
print_bm_table(bm_minor, "MINOR")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 5: COMPOSITE MODEL + R²
# ══════════════════════════════════════════════════════════════════════════════
#
# The model price at any time t is:
#   log10(price) = log10(support(t))  +  Σ bubble_shape_i(t)
# which in linear price is:
#   price = support(t) × 10^[Σ bubble_shape_i(t)]
#
# R² is computed on the full daily time series in log10-price space.
# We compare two models: support-only (baseline) and support + all bubbles.
# ══════════════════════════════════════════════════════════════════════════════

# Dense plotting grid (years since genesis)
years_plot_bm       = np.linspace(PLOT_YEARS_MIN, PLOT_YEARS_MAX, PLOT_GRID_POINTS)
log_t_plot_bm       = np.log10(years_plot_bm)
log_support_plot_bm = intercept_sup + slope_sup * log_t_plot_bm   # log10(support)
support_plot_bm     = 10 ** log_support_plot_bm                    # support in USD


def bm_total_bubble(years, params_list, budget=None):
    """
    Sum bubble_shape contributions (log-excess) from all bubbles in params_list.

    If budget is provided (array, same length as years), applies sequential
    subtraction: bubbles are sorted chronologically by t_rise and each bubble's
    contribution at any time t is capped to the remaining budget at t.  This
    mirrors the sequential residual fitting and prevents overlapping tails from
    stacking beyond what the data supports.

    Without a budget, falls back to a plain additive sum (original behaviour).
    Returns an array of shape (len(years),).
    """
    if not params_list:
        return np.zeros(len(years))
    if budget is None:
        total = np.zeros(len(years))
        for bp in params_list:
            total += bubble_shape(years, bp['t_rise'], bp['r'],
                                  bp['t_plateau'], bp['t_decay'], bp['d'],
                                  bp.get('plat_pow', 0.0))
        return total
    # Sequential capped sum
    ordered  = sorted(params_list, key=lambda b: b['t_rise'])
    total    = np.zeros(len(years))
    remaining = np.maximum(0.0, budget.copy())
    for bp in ordered:
        contrib   = bubble_shape(years, bp['t_rise'], bp['r'],
                                 bp['t_plateau'], bp['t_decay'], bp['d'],
                                 bp.get('plat_pow', 0.0))
        contrib   = np.minimum(contrib, remaining)
        total    += contrib
        remaining = np.maximum(0.0, remaining - contrib)
    return total


# ── Budget for sequential-cap composite ──────────────────────────────────────
# Maximum log-excess ever observed in the historical data.  No single time
# point in the composite is allowed to exceed this.
_hist_K_max   = float(np.max(np.maximum(0.0, log_excess_fit)))
_budget_plot  = np.full(len(years_plot_bm), _hist_K_max) if CAP_COMPOSITE_OVERLAP else None
_budget_all   = np.full(len(years_all),     _hist_K_max) if CAP_COMPOSITE_OVERLAP else None

# ── Plotting-grid composites ─────────────────────────────────────────────────
# Process all fitted bubbles together so the budget is shared across maj/min.
_all_fitted   = sorted(bm_major + bm_minor, key=lambda b: b['t_rise'])
bm_total_plot = bm_total_bubble(years_plot_bm, _all_fitted, _budget_plot)
bm_maj_plot   = bm_total_bubble(years_plot_bm, bm_major)   # uncapped, for decomp plots
bm_min_plot   = bm_total_bubble(years_plot_bm, bm_minor)   # uncapped, for decomp plots
bm_composite  = 10 ** (log_support_plot_bm + bm_total_plot)   # USD price curve

# ── R² on the full daily time series ─────────────────────────────────────────
total_all_bm     = bm_total_bubble(years_all, _all_fitted, _budget_all)
composite_all_bm = log_support_all + total_all_bm
ss_tot           = np.sum((log_p_all - np.mean(log_p_all)) ** 2)
bm_r2_support    = 1 - np.sum((log_p_all - log_support_all)  ** 2) / ss_tot
bm_r2_comp       = 1 - np.sum((log_p_all - composite_all_bm) ** 2) / ss_tot

print(f"\n{'='*80}\nMODEL FIT QUALITY\n{'='*80}")
print(f"  R² support only:          {bm_r2_support:.6f}")
print(f"  R² support + all bubbles: {bm_r2_comp:.6f}   ΔR² = {bm_r2_comp - bm_r2_support:.6f}")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 6: PREDICT FUTURE BUBBLES
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*80}")
print("STEP 6: PREDICTING FUTURE BUBBLES")
print(f"{'='*80}")


def predict_future_bubbles(hist_params, n_future, extrap_weights,
                            use_interval_trend, label='', default_interval=3.8,
                            min_interval=0.15, mode='extrap', last_n=None):
    """Project future bubble parameters from historical bubbles.

    mode='avg'
        Every predicted bubble gets the weighted average of the last `last_n`
        historical bubbles.  All predictions are identical in shape; only the
        start time advances by the average interval.  Conservative / stable.

    mode='extrap'
        Parameters are linearly extrapolated along their historical trend
        (r and d log-linearly; K, dur_plateau, plat_pow linearly), so each
        successive prediction follows the fitted slope.

    last_n
        If set, only the most-recent N historical bubbles are used for the
        average or extrapolation.  None = use all.

    Weights layout per bubble (chronological):
        [r_rise, d_decay, K, interval, dur_plateau, plat_pow]
    K is fitted directly; dur_rise is derived as K/r to avoid blowup.
    """
    future = []
    if last_n is not None:
        hist_params = hist_params[-last_n:]
    n_hist = len(hist_params)
    if n_hist < 1 or n_future < 1:
        if n_future > 0:
            print(f"  [{label}] No historical bubbles — cannot predict.")
        return future

    n_params   = 6
    expected_w = n_hist * n_params
    use_w = (extrap_weights is not None and len(extrap_weights) == expected_w)
    if extrap_weights is not None and not use_w:
        print(f"  [{label}] Weight length mismatch: expected {expected_w}, "
              f"got {len(extrap_weights)}. Using uniform.")
    w = (np.array(extrap_weights, dtype=float).reshape(n_hist, n_params)
         if use_w else np.ones((n_hist, n_params)))

    starts        = [b['t_rise'] for b in hist_params]
    intervals_arr = np.array([starts[i+1] - starts[i] for i in range(len(starts) - 1)]
                             if n_hist >= 2 else [default_interval])
    intv_weights  = w[:len(intervals_arr), 3]

    vals_r  = np.array([b['r']           for b in hist_params])
    vals_d  = np.array([b['d']           for b in hist_params])
    vals_K  = np.array([b['K']           for b in hist_params])
    vals_dp = np.array([b['dur_plateau'] for b in hist_params])
    vals_pp = np.array([b.get('plat_pow', 0.0) for b in hist_params])

    def wextrap(vals, wi, target):
        x = np.arange(len(vals), dtype=float)
        c = np.polyfit(x, vals, 1, w=wi) if len(vals) >= 2 else [0.0, float(vals[0])]
        return float(np.polyval(c, target)), c

    def wavg(vals, wi):
        return float(np.average(vals, weights=wi))

    last_n_str = f'last {last_n}' if last_n is not None else 'all'
    print(f"\n  [{label}] mode={mode}  using {n_hist} bubbles ({last_n_str})")
    for lbl2, vals, wi, log_sc in [
            ('r',           vals_r,  w[:, 0], True),
            ('d',           vals_d,  w[:, 1], True),
            ('K',           vals_K,  w[:, 2], True),
            ('dur_plateau', vals_dp, w[:, 4], False),
            ('plat_pow',    vals_pp, w[:, 5], False)]:
        fit_v = np.log(np.maximum(vals, 1e-9)) if log_sc else vals
        data_str = ['%.3f' % v for v in vals]
        if mode == 'extrap':
            _, c = wextrap(fit_v, wi, 0)
            suffix = ' [log-lin]' if log_sc else ''
            print(f"    {lbl2:12s}: slope={c[0]:+.4f}  data={data_str}{suffix}")
        else:
            avg_v = wavg(fit_v, wi)
            disp  = np.exp(avg_v) if log_sc else avg_v
            print(f"    {lbl2:12s}: avg={disp:.4f}  data={data_str}")
    if use_interval_trend and len(intervals_arr) >= 2:
        _, c = wextrap(intervals_arr, intv_weights, 0)
        print(f"    {'interval':12s}: slope={c[0]:+.4f}  "
              f"data={['%.3f'%v for v in intervals_arr]}  [trend]")
    else:
        wt_avg = float(np.average(intervals_arr, weights=intv_weights))
        print(f"    {'interval':12s}: weighted avg={wt_avg:.3f}  "
              f"data={['%.3f'%v for v in intervals_arr]}  [avg]")

    # Pre-compute avg-mode fixed parameters (same for every prediction)
    if mode == 'avg':
        fixed_r  = np.exp(wavg(np.log(np.maximum(vals_r, 1e-9)), w[:, 0]))
        fixed_d  = np.exp(wavg(np.log(np.maximum(vals_d, 1e-9)), w[:, 1]))
        fixed_K  = np.exp(wavg(np.log(np.maximum(vals_K, 1e-9)), w[:, 2]))
        fixed_dp = wavg(vals_dp, w[:, 4])
        fixed_pp = float(np.clip(wavg(vals_pp, w[:, 5]), -PLAT_POW_RANGE, PLAT_POW_RANGE))

    last_start = starts[-1]
    for j in range(n_future):
        tgt = n_hist + j
        if mode == 'avg':
            pred_r, pred_d, pred_K, pred_dp, pred_pp = \
                fixed_r, fixed_d, fixed_K, fixed_dp, fixed_pp
        else:
            _lr, _ = wextrap(np.log(np.maximum(vals_r, 1e-9)), w[:, 0], tgt)
            pred_r = np.exp(_lr)
            _ld, _ = wextrap(np.log(np.maximum(vals_d, 1e-9)), w[:, 1], tgt)
            pred_d = np.exp(_ld)
            _lk, _ = wextrap(np.log(np.maximum(vals_K, 1e-9)), w[:, 2], tgt)
            pred_K = np.exp(_lk)
            pred_dp, _ = wextrap(vals_dp, w[:, 4], tgt)
            pred_pp, _ = wextrap(vals_pp, w[:, 5], tgt)
            pred_pp    = float(np.clip(pred_pp, -PLAT_POW_RANGE, PLAT_POW_RANGE))

        if use_interval_trend and len(intervals_arr) >= 2:
            pred_intv, _ = wextrap(intervals_arr, intv_weights, len(intervals_arr) + j)
        else:
            pred_intv = float(np.average(intervals_arr, weights=intv_weights))

        pred_K    = max(pred_K,    0.01)
        pred_dp   = max(pred_dp,   0.0)
        pred_intv = max(pred_intv, min_interval)

        pred_tr = (last_start if j == 0 else future[-1]['t_rise']) + pred_intv
        if j == 0:
            while pred_tr <= years_all[-1]:
                pred_tr += pred_intv
        pred_dr  = max(pred_K / pred_r, 0.02)
        pred_tpl = pred_tr + pred_dr
        pred_tdc = pred_tpl + pred_dp
        pred_K_end  = (pred_K + pred_pp * np.log10(max(pred_tdc / pred_tpl, 1e-12))
                       if pred_dp > 0 else pred_K)
        peak_K_disp = max(pred_K, pred_K_end)

        sup_at_pk = A_sup * pred_tpl ** B_sup
        peak_px   = sup_at_pk * 10 ** peak_K_disp

        fb = {
            't_rise': pred_tr, 't_start': pred_tr,
            'r': pred_r, 't_plateau': pred_tpl,
            't_decay': pred_tdc, 'd': pred_d,
            'K': peak_K_disp, 'K_peak': pred_K,
            'K_end': pred_K_end, 'plat_pow': pred_pp,
            'dur_rise': pred_dr, 'dur_plateau': pred_dp,
            'interval': pred_intv,
            'date_start': genesis + pd.Timedelta(days=pred_tr  * 365.25),
            'date_plat':  genesis + pd.Timedelta(days=pred_tpl * 365.25),
            'date_decay': genesis + pd.Timedelta(days=pred_tdc * 365.25),
        }
        future.append(fb)
        intv_mode = 'trend' if use_interval_trend else 'avg'
        print(f"    Predicted {label} #{n_hist+j+1}:  "
              f"t_rise={pred_tr:.2f} yr  ({fb['date_start'].strftime('%Y-%m-%d')})  "
              f"interval={pred_intv:.2f} yr [{intv_mode}]  "
              f"K={peak_K_disp:.3f} ({10**peak_K_disp:.1f}×)  "
              f"plat_pow={pred_pp:+.2f}  peak≈${peak_px:,.0f}")
    return future


bm_future_major = predict_future_bubbles(
    bm_major, N_PREDICT_MAJOR,
    MAJOR_EXTRAP_WEIGHTS, MAJOR_INTERVAL_USE_TREND,
    label='MAJOR', default_interval=MAJOR_INTERVAL_YR, min_interval=1.4,
    mode=PREDICT_MODE, last_n=PREDICT_LAST_N_MAJOR)

bm_future_minor = []
if N_PREDICT_MINOR > 0:
    bm_future_minor = predict_future_bubbles(
        bm_minor, N_PREDICT_MINOR,
        MINOR_EXTRAP_WEIGHTS, MINOR_INTERVAL_USE_TREND,
        label='MINOR', default_interval=MAJOR_INTERVAL_YR, min_interval=0.15,
        mode=PREDICT_MODE, last_n=PREDICT_LAST_N_MINOR)

# Composite including predicted bubbles (for plotting)
# When capping, continue from the remaining budget after the fitted bubbles.
_future_all     = sorted(bm_future_major + bm_future_minor, key=lambda b: b['t_rise'])
if CAP_COMPOSITE_OVERLAP and _future_all:
    # Re-derive the remaining budget after all fitted bubbles consumed their share.
    _budget_future  = np.full(len(years_plot_bm), _hist_K_max)
    for bp in _all_fitted:
        _c = bubble_shape(years_plot_bm, bp['t_rise'], bp['r'],
                          bp['t_plateau'], bp['t_decay'], bp['d'],
                          bp.get('plat_pow', 0.0))
        _budget_future = np.maximum(0.0, _budget_future - np.minimum(_c, _budget_future))
    bm_future_total = bm_total_plot.copy()
    _rem = _budget_future
    for fb in _future_all:
        _c = bubble_shape(years_plot_bm, fb['t_rise'], fb['r'],
                          fb['t_plateau'], fb['t_decay'], fb['d'],
                          fb.get('plat_pow', 0.0))
        _c = np.minimum(_c, _rem)
        bm_future_total += _c
        _rem = np.maximum(0.0, _rem - _c)
else:
    bm_future_total = bm_total_plot.copy()
    for fb in _future_all:
        bm_future_total += bubble_shape(years_plot_bm, fb['t_rise'], fb['r'],
                                        fb['t_plateau'], fb['t_decay'], fb['d'],
                                        fb.get('plat_pow', 0.0))
bm_composite_future = 10 ** (log_support_plot_bm + bm_future_total)

print(f"\n{'='*80}")


# ══════════════════════════════════════════════════════════════════════════════
# PLOT HELPERS
# ══════════════════════════════════════════════════════════════════════════════
_exp_bm       = np.arange(-2, 10)
_maj_ticks_bm = 10.0 ** _exp_bm
_min_ticks_bm = [s * 10.0 ** e for e in _exp_bm for s in range(2, 10)]
_every_yr_bm  = np.arange(1, 43)
_xtv_bm       = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 35, 40]
_xtl_bm       = [f"{y}\n{2009+y}" for y in _xtv_bm]
_xtv_lin_bm   = list(range(1, 18, 2))
_xtl_lin_bm   = [f"{y}\n({2009+y})" for y in _xtv_lin_bm]


def setup_loglog_bm(ax):
    """
    Configure ax for a log-log Bitcoin price chart.

    Both axes are logarithmic.  Y-axis shows USD price with dollar formatting;
    X-axis shows years since genesis with labels at round years.  Minor gridlines
    mark every integer year so individual halving cycles are visible.
    """
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.yaxis.set_major_locator(FixedLocator(_maj_ticks_bm))
    ax.yaxis.set_major_formatter(StrMethodFormatter('${x:,.0f}'))
    ax.yaxis.set_minor_locator(FixedLocator(_min_ticks_bm))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xticks(_xtv_bm); ax.set_xticklabels(_xtl_bm, fontsize=8)
    ax.xaxis.set_minor_locator(FixedLocator(_every_yr_bm))
    ax.grid(which='major', color=GRID_MAJOR_COLOR, lw=0.6, alpha=0.6)
    ax.grid(which='minor', color=GRID_MINOR_COLOR, lw=0.3, alpha=0.4)
    ax.set_facecolor(PLOT_BG_COLOR)


def save_plot_bm(basename):
    """Save the current figure as both vector (SVG) and raster (JPG at 200 dpi)."""
    plt.savefig(f'{basename}.svg', format='svg', bbox_inches='tight')
    plt.savefig(f'{basename}.jpg', format='jpg', bbox_inches='tight', dpi=200)
    print(f"  Saved {basename}.svg/.jpg")


def set_linear_xticks_bm(ax):
    """Apply the standard linear-axis year labels to ax."""
    ax.set_xticks(_xtv_lin_bm)
    ax.set_xticklabels(_xtl_lin_bm, fontsize=8)


def draw_bubbles_bm(ax, params, colors, ls='-', lw=1.2, alpha=0.65, label_prefix=''):
    """Draw individual bubble price curves on ax using the dense plotting grid."""
    if not params or not colors:
        return
    n_c = len(colors)
    for i, bp in enumerate(params):
        bub = bubble_shape(years_plot_bm, bp['t_rise'], bp['r'],
                           bp['t_plateau'], bp['t_decay'], bp['d'],
                           bp.get('plat_pow', 0.0))
        mask = bub > 0.001
        if mask.any():
            ax.plot(years_plot_bm[mask], 10 ** (log_support_plot_bm[mask] + bub[mask]),
                    color=colors[i % n_c], ls=ls, lw=lw, alpha=alpha,
                    label=f"{label_prefix}{i+1}" if i < 8 else None)


# ══════════════════════════════════════════════════════════════════════════════
# PLOT A: LOG-LOG PRICE CHART WITH COMPOSITE MODEL
# Both axes are logarithmic.  On a log-log chart a pure power law (support line)
# appears as a straight line.  Each bubble's individual price trajectory is
# plotted (major = solid, minor = dotted); the thick red composite is their sum
# translated back to USD: price = support(t) × 10^[Σ bubble_shape_i(t)].
# Predicted bubbles are shown with dashed lines.
# ══════════════════════════════════════════════════════════════════════════════

In [ ]:
# Reset to inline backend in case interactive cell (cell 4) was run first
try:
    _ip = get_ipython()  # noqa: F821
    if _ip is not None:
        _ip.run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# ══════════════════════════════════════════════════════════════════════════════
# QUANTILE REGRESSION CHANNEL MODEL
# Standalone cell — self-contained: loads data, fits power-law quantile
# regressions in log-log space, then produces:
#   Plot 1 — semi-log channels  (log price, linear time) + OLS + R²
#   Plot 2 — log-log channels   (log price, log time)    + OLS + R²
#   Plot 3 — zoomed semi-log    (2025-2035)               + OLS + R²
#   Plot 4 — bubble model overlay (requires cell 0 to have been run)
#   Table 1 — price at each quantile for years 2026-2036
#   Table 2 — CAGR heatmap from chosen entry scenarios
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as mcm
from matplotlib.ticker import (FixedLocator, MultipleLocator, StrMethodFormatter,
                                NullFormatter, LogFormatter, FuncFormatter)
from scipy.stats import linregress
from statsmodels.regression.quantile_regression import QuantReg
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

csv_path     = './BitcoinPricesDaily.csv'
GENESIS_DATE = pd.Timestamp('2009-07-25')
FIT_MIN_DATE = '2010-07-17'

#QR_QUANTILES = [0.01, 0.1, 0.05, 0.15, 0.20, 0.25, 0.5, 0.8]
QR_QUANTILES = [0.00001, 0.0001, 0.001, 0.01, 0.05, 0.1, 0.15, 0.20, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99, 0.999, 0.9999, 0.99999]
#QR_QUANTILES = [0.001, 0.05, 0.1, 0.15, 0.20, 0.25]
# Quantiles to fit and plot.  Adjacent pairs are shaded.
# Must be strictly increasing, values in (0, 1).

EXTRAP_YEARS = 20
# Years to extrapolate beyond today on all plots.

TABLE_YEARS  = list(range(2025, 2041))
# Calendar years for the price table rows and CAGR exit years.

ZOOM_YEAR_LO  = 2025      # zoomed chart left bound (calendar year)
ZOOM_YEAR_HI  = 2038      # zoomed chart right bound (calendar year)
ZOOM_PRICE_LO = 4e4       # zoomed chart lower price limit (USD)
ZOOM_PRICE_HI = 1.75e6       # zoomed chart upper price limit (USD)

CAGR_FROM_SCENARIOS = [
    (2026, 0.001),   # 
    (2026, 0.01),   # 
    (2026, 0.05),   # 
    (2025, 0.85),   # entry: 5th-pct price in 2026 (buying at a dip)
#    (2027, 0.5),   # entry: 5th-pct price in 2027 (buying at a dip)
#    (2030, 0.90),   # entry: 5th-pct price in 2036 (buying at a dip)
]

CAGR_HEATMAP_FONTSIZE = 6
# Font size for the annotated numbers inside each CAGR heatmap cell.
# Reduce if cells are crowded (e.g. 7 or 8); increase for larger figures.

# ── Data-scaled heatmap colour variables ─────────────────────────────────
CAGR_COLOR_MIN  = '#4393C3'   # blue  — lowest  CAGR in the scaled heatmap
CAGR_COLOR_MID  = '#F7F7F7'   # white — midpoint between min and max
CAGR_COLOR_MAX  = '#E08030'   # amber — highest CAGR in the scaled heatmap
CAGR_GRAD_STEPS = 24          # discrete colour bands on each side of midpoint
CAGR_SEG_B1     =  5.0      # % CAGR — boundary 1: blue→white ends here
CAGR_SEG_B2     = 16.0      # % CAGR — boundary 2: orange→red starts here
CAGR_SEG_C_LO   = '#2166AC' # deep blue     at data-min
CAGR_SEG_C_MID1 = '#F7F7F7' # white         at CAGR_SEG_B1
CAGR_SEG_C_MID2 = '#FF8C00' # bright orange at CAGR_SEG_B2
CAGR_SEG_C_HI   = '#CC1100' # bright red    at data-max

# ── Futuristic + Typewriter colour theme ──────────────────────────────
# Quantile colormap: deep navy -> electric blue -> cyan -> lime -> neon gold
QR_CMAP = mcolors.LinearSegmentedColormap.from_list('print_bands', [
    '#2040A0', '#208080', '#208040', '#806020', '#A03020'])

# Diverging colormap for CAGR: neon red -> dark navy -> electric green
MIL_DIV_CMAP = mcolors.LinearSegmentedColormap.from_list('cb_div', [
    '#4393C3',   # CB-safe medium blue  (negative / low CAGR)
    '#F7F7F7',   # near-white neutral centre
    '#E08030'])  # CB-safe amber orange (positive / high CAGR)

CHART_FONT       = 'Special Elite'  # old typewriter font
PLOT_BG_COLOR    = '#FFFFFF'   # white
SPINE_COLOR      = '#888888'   # medium gray frame
TEXT_COLOR       = '#222222'   # near black
TITLE_COLOR      = '#1A3060'   # dark navy
LEGEND_BG_COLOR  = '#F5F5F5'   # very light gray legend bg
DATA_COLOR       = '#606060'   # medium gray scatter
DATA_PT_SIZE     = 16         # scatter point size for all standard charts
DATA_PT_SIZE_ZOOM = 32          # scatter point size for the zoomed chart
OLS_COLOR        = '#1A1A1A'   # near black OLS mean line
GRID_MAJOR_COLOR = '#BBBBBB'   # medium gray major grid
GRID_MINOR_COLOR = '#E8E8E8'   # very light gray minor grid
TODAY_COLOR      = '#CC2200'   # dark red today line
BUBBLE_COLOR     = '#CC3300'   # dark red-orange bubble composite

# Colorblind-friendly Okabe-Ito colour cycle and line-style cycle.
# Auto-assigned by index to whatever quantiles are in QR_QUANTILES,
# so adding/removing quantiles never causes a KeyError.
_CB_COLORS = [
    '#1855B0',   # medium blue
    '#167050',   # dark green
    '#B01020',   # dark red
    '#6B3090',   # dark purple
    '#A06000',   # dark amber
    '#006878',   # dark teal
    '#A02060',   # dark magenta
    '#404040',   # dark charcoal
]
_CB_STYLES = [
    '-',               # solid
    '--',              # dashed
    (0, (5, 2)),       # long dash
    (0, (1, 2)),       # dotted
    '-.',              # dash-dot
    (0, (3,2,1,2)),   # dash-dot-dot
    (0, (5,2,1,2)),   # long-dash-dot
    ':',               # dense dotted
]

PRICE_YMIN = 0.01
PRICE_YMAX = 1e8
PLOT_YEARS_MIN = 1.0
PLOT_GRID_PTS  = 8000
CHART_DPI      = 100
# Resolution of saved JPG files in dots-per-inch.
# Higher = larger file, more detail. 150 = draft, 200 = default, 300 = print.

CHART_W        = 15*0.5     # width  (inches) for standard charts
CHART_H        = 9*0.5   # height (inches) for standard charts
CHART_ZOOM_W   = 13*0.5     # width  (inches) for the zoomed chart
CHART_ZOOM_H   = 8*0.5      # height (inches) for the zoomed chart


# ══════════════════════════════════════════════════════════════════════════════
# APPLY MILITARY MATPLOTLIB THEME
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.font_manager as _fm
_fm._load_fontmanager(try_read_cache=False)   # pick up Special Elite & 1942 report

plt.rcParams.update({
    'figure.facecolor':   PLOT_BG_COLOR,
    'axes.facecolor':     PLOT_BG_COLOR,
    'axes.edgecolor':     SPINE_COLOR,
    'axes.labelcolor':    TEXT_COLOR,
    'axes.titlecolor':    TITLE_COLOR,
    'xtick.color':        TEXT_COLOR,
    'ytick.color':        TEXT_COLOR,
    'text.color':         TEXT_COLOR,
    'legend.facecolor':   LEGEND_BG_COLOR,
    'legend.edgecolor':   SPINE_COLOR,
    'legend.labelcolor':  TEXT_COLOR,
    'legend.framealpha':  0.88,
    'font.family':        [CHART_FONT],
})

# ══════════════════════════════════════════════════════════════════════════════
# LOAD + PREPARE DATA
# ══════════════════════════════════════════════════════════════════════════════

try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    raise FileNotFoundError(f"CSV not found: {csv_path}")

df.columns = df.columns.str.strip()
date_col  = next((c for c in df.columns if 'date'  in c.lower()), df.columns[0])
price_col = next((c for c in df.columns if any(k in c.lower()
                  for k in ['close', 'price', 'usd'])),           df.columns[1])

df = (df[[date_col, price_col]]
      .rename(columns={date_col: 'date', price_col: 'price'})
      .assign(date=lambda d: pd.to_datetime(d['date']))
      .sort_values('date').reset_index(drop=True))
df = df[df['price'] > 0].dropna()

df['years']     = (df['date'] - GENESIS_DATE).dt.days / 365.25
df['log_years'] = np.log10(df['years'])
df['log_price'] = np.log10(df['price'])

fit_mask = df['date'] >= pd.Timestamp(FIT_MIN_DATE)
df_fit   = df[fit_mask].copy()
X_fit    = np.column_stack([np.ones(len(df_fit)), df_fit['log_years'].values])

today_yr      = (pd.Timestamp.today() - GENESIS_DATE).days / 365.25
PLOT_YEARS_MAX = today_yr + EXTRAP_YEARS + 0.5   # extends 20 yr past today

print(f"Data  : {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()}"
      f"  ({len(df):,} rows)")
print(f"Fit   : {FIT_MIN_DATE} onwards  ({len(df_fit):,} rows)")
print(f"Today : year {today_yr:.2f} since genesis"
      f"  |  extrapolating to year {PLOT_YEARS_MAX:.1f} (~{2009+PLOT_YEARS_MAX:.0f})")


# ══════════════════════════════════════════════════════════════════════════════
# FIT QUANTILE REGRESSIONS  +  OLS MEAN REGRESSION
# Model: log10(price) = intercept + slope · log10(t_years)   [power law]
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nFitting OLS + {len(QR_QUANTILES)} quantile regressions ...")

# OLS (mean) regression
ols_slope, ols_intercept, ols_r, _, _ = linregress(
    df_fit['log_years'].values, df_fit['log_price'].values)
ols_r2 = ols_r ** 2
print(f"  OLS: log10(P) = {ols_intercept:+.4f} + {ols_slope:.4f}·log10(t)"
      f"   R² = {ols_r2:.4f}")

# Per-quantile R² (OLS formula applied to QR predictions — measures how well
# each channel tracks the conditional mean of the data)
ss_tot_fit = np.sum((df_fit['log_price'].values - df_fit['log_price'].mean()) ** 2)

qr_fits = {}
for q in QR_QUANTILES:
    res  = QuantReg(df_fit['log_price'].values, X_fit).fit(q=q, max_iter=2000)
    pred = res.params[0] + res.params[1] * df_fit['log_years'].values
    r2_q = 1 - np.sum((df_fit['log_price'].values - pred) ** 2) / ss_tot_fit
    qr_fits[q] = {'intercept': float(res.params[0]),
                  'slope':     float(res.params[1]),
                  'r2':        float(r2_q)}
    print(f"  Q{int(q*100):02d}: log10(P) = {res.params[0]:+.4f}"
          f" + {res.params[1]:.4f}·log10(t)   R² = {r2_q:.4f}")



In [ ]:
# ── Export model data (lean pkl — computation keys only) ─────────────────────
import pickle as _pkl, os as _os, numpy as _np

_export_dir = _os.path.join(_os.path.dirname(_os.path.abspath('SP.ipynb')), 'btc_app')
_os.makedirs(_export_dir, exist_ok=True)

# Precompute bubble composite for N = 0 … len(future_all) future bubbles
_fut_all   = sorted(bm_future_major + bm_future_minor, key=lambda b: b['t_rise'])
_max_n     = len(_fut_all)
_comp_by_n = _np.zeros((_max_n + 1, len(years_plot_bm)))

for _n in range(_max_n + 1):
    _subset = _fut_all[:_n]
    if CAP_COMPOSITE_OVERLAP:
        _budget = _np.full(len(years_plot_bm), _hist_K_max)
        for _bp in _all_fitted:
            _c = bubble_shape(years_plot_bm, _bp['t_rise'], _bp['r'],
                              _bp['t_plateau'], _bp['t_decay'], _bp['d'],
                              _bp.get('plat_pow', 0.0))
            _budget = _np.maximum(0.0, _budget - _np.minimum(_c, _budget))
        _tot = bm_total_plot.copy()
        _rem = _budget.copy()
        for _fb in _subset:
            _c   = bubble_shape(years_plot_bm, _fb['t_rise'], _fb['r'],
                                _fb['t_plateau'], _fb['t_decay'], _fb['d'],
                                _fb.get('plat_pow', 0.0))
            _cap = _np.minimum(_c, _rem)
            _tot = _tot + _cap
            _rem = _np.maximum(0.0, _rem - _cap)
    else:
        _tot = bm_total_plot.copy()
        for _fb in _subset:
            _tot += bubble_shape(years_plot_bm, _fb['t_rise'], _fb['r'],
                                 _fb['t_plateau'], _fb['t_decay'], _fb['d'],
                                 _fb.get('plat_pow', 0.0))
    _comp_by_n[_n] = 10.0 ** (log_support_plot_bm + _tot)

# Serialise lean model (13 computation keys — no visual config)
_model = {
    'qr_fits':         {str(k): dict(v) for k, v in qr_fits.items()},
    'QR_QUANTILES':    list(QR_QUANTILES),
    'ols_intercept':   float(ols_intercept),
    'ols_slope':       float(ols_slope),
    'GENESIS_DATE':    str(GENESIS_DATE.date()),

    'years_plot_bm':   list(years_plot_bm),
    'support_plot_bm': list(support_plot_bm),
    'bm_comp_by_n':    [c.tolist() for c in _comp_by_n],
    'bm_r2_comp':      float(bm_r2_comp),
    'bm_n_future_max': int(_max_n),

    'price_dates':     df['date'].dt.strftime('%Y-%m-%d').tolist(),
    'price_years':     df['years'].tolist(),
    'price_prices':    df['price'].tolist(),
}

_out = _os.path.join(_export_dir, 'model_data.pkl')
with open(_out, 'wb') as _f:
    _pkl.dump(_model, _f, protocol=4)
print(f"Model data exported \u2192 {_out}  ({_os.path.getsize(_out)//1024} KB)")
del _pkl, _os, _np, _export_dir, _fut_all, _max_n, _comp_by_n, _model, _out
del _n, _subset, _budget, _bp, _c, _tot, _rem, _fb, _cap
